# Data Mining Assignment 4 – Classification

Imports

In [143]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 99

Datasets

In [144]:
df_income = pd.read_csv('income.csv')
df_income_test = pd.read_csv('income_test.csv')
df_predictions_template = pd.read_csv('predictions_template.csv')

## Task 1: Building and Evaluating Classification Models

Inspect dataset

In [145]:
# missing values
print("missing values")
print(df_income.isnull().sum())
# head
df_income.head()

missing values
age                            0
workclass                      0
education                      0
marital status                 0
occupation                     0
workinghours                   0
sex                            0
ability to speak english    8597
gave birth this year        7058
income                         0
dtype: int64


,age,workclass,education,marital status,occupation,workinghours,sex,ability to speak english,gave birth this year,income
0,52,self employed,16,Widowed,Transport,50,Male,NaN,NaN,high
1,60,private,20,Divorced,Healthcare/Medical Services,30,Female,NaN,NaN,low
2,64,private,21,Divorced,Management/Business,40,Male,NaN,NaN,low
3,64,private,17,Husband,Transport,40,Male,NaN,NaN,low
4,31,private,15,Husband,Transport,40,Male,NaN,NaN,low


Fill missing values

In [146]:
for data in [df_income, df_income_test]:
    # treating NaN as 'No' for birth records
    data['gave birth this year'] = data['gave birth this year'].fillna('No')

    # treating NaN as 'Yes' for English ability
    data['ability to speak english'] = data['ability to speak english'].fillna('Yes')

Encoding of Features

In [147]:
# map binary features to 1/0
binary_mapping = {'Male': 1, 'Female': 0, 'Yes': 1, 'No': 0}
binary_cols = ['sex', 'ability to speak english', 'gave birth this year']

for col in binary_cols:
    df_income[col] = df_income[col].map(binary_mapping)
    df_income_test[col] = df_income_test[col].map(binary_mapping)

df_income['income'] = df_income['income'].map({'high': 1, 'low': 0})

# one-hot encoding of nominal columns
nominal_cols = ['workclass', 'marital status', 'occupation']
df_income = pd.get_dummies(df_income, columns=nominal_cols)
df_income_test = pd.get_dummies(df_income_test, columns=nominal_cols)

# convert columns to 1/0
fix_cols = df_income.select_dtypes(include=['bool']).columns.tolist() + ['ability to speak english']
df_income[fix_cols] = df_income[fix_cols].fillna(0).astype(int)
df_income_test[fix_cols] = df_income_test[fix_cols].fillna(0).astype(int)
df_income[fix_cols] = df_income[fix_cols].astype(int)
df_income_test[fix_cols] = df_income_test[fix_cols].astype(int)

# show head
df_income.head()

,age,education,workinghours,sex,ability to speak english,gave birth this year,income,workclass_governmental,workclass_no paid work,workclass_private,...,occupation_Management/Business,occupation_Military Services,occupation_Office/Administrative Support,occupation_Production/Assembly,occupation_Protective Services,occupation_Repair/Maintenance,occupation_Sales,"occupation_Science, Engineering, Technology",occupation_Service/Hospitality,occupation_Transport
0,52,16,50,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,60,20,30,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,64,21,40,1,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
3,64,17,40,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
4,31,15,40,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1


Normalizing

In [148]:
# initialize the scaler
num_cols = ['age', 'education', 'workinghours']
scaler = StandardScaler()

# fit on training data and transform both
df_income[num_cols] = scaler.fit_transform(df_income[num_cols])
df_income_test[num_cols] = scaler.transform(df_income_test[num_cols])

df_income.head()

,age,education,workinghours,sex,ability to speak english,gave birth this year,income,workclass_governmental,workclass_no paid work,workclass_private,...,occupation_Management/Business,occupation_Military Services,occupation_Office/Administrative Support,occupation_Production/Assembly,occupation_Protective Services,occupation_Repair/Maintenance,occupation_Sales,"occupation_Science, Engineering, Technology",occupation_Service/Hospitality,occupation_Transport
0,0.552525,-0.715511,0.864398,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1.076094,0.529517,-0.740484,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1.337879,0.840774,0.061957,1,1,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
3,1.337879,-0.404254,0.061957,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
4,-0.821844,-1.026767,0.061957,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1


Split target

In [149]:
# Define X and y
X = df_income.drop('income', axis=1)
y = df_income['income']

# Split into Training and Validation sets (80/20 split)
# This setup is essential for "explicitly investigating overfitting"
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)

print(f"Training features: {X_train.shape}")
print(f"Validation features: {X_val.shape}")

Training features: (7200, 34)
Validation features: (1800, 34)


Model 1: Logistic Regression (Baseline)

In [150]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Initialize and train
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# Check for overfitting
train_preds = log_model.predict(X_train)
val_preds = log_model.predict(X_train) # Wait, this should be X_val

print(f"Logistic Regression Training Accuracy: {accuracy_score(y_train, train_preds):.4f}")
print(f"Logistic Regression Validation Accuracy: {accuracy_score(y_val, log_model.predict(X_val)):.4f}")

Logistic Regression Training Accuracy: 0.7872
Logistic Regression Validation Accuracy: 0.7822


Model 2: Random Forest (Ensemble + Overfitting Technique)

In [151]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# We use GridSearchCV to optimize hyperparameters as required
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None], # 'None' often leads to overfitting
    'min_samples_leaf': [1, 4]   # Higher values reduce overfitting
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=3, scoring='accuracy')
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print(f"Best Parameters: {rf_grid.best_params_}")

Best Parameters: {'max_depth': 20, 'min_samples_leaf': 4, 'n_estimators': 100}


Evaluation

In [152]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

def evaluate_model(model, X_v, y_v, name):
    preds = model.predict(X_v)
    probs = model.predict_proba(X_v)[:, 1]

    print(f"--- Evaluation for {name} ---")
    print(f"Accuracy: {accuracy_score(y_v, preds):.4f}")
    print(f"AUC Score: {roc_auc_score(y_v, probs):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_v, preds))
    print("-" * 30)

# Evaluate both models
evaluate_model(log_model, X_val, y_val, "Logistic Regression")
evaluate_model(best_rf, X_val, y_val, "Optimized Random Forest")

--- Evaluation for Logistic Regression ---
Accuracy: 0.7822
AUC Score: 0.8595

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.87      0.84      1157
           1       0.73      0.62      0.67       643

    accuracy                           0.78      1800
   macro avg       0.77      0.75      0.75      1800
weighted avg       0.78      0.78      0.78      1800

------------------------------
--- Evaluation for Optimized Random Forest ---
Accuracy: 0.7872
AUC Score: 0.8586

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84      1157
           1       0.74      0.63      0.68       643

    accuracy                           0.79      1800
   macro avg       0.77      0.75      0.76      1800
weighted avg       0.78      0.79      0.78      1800

------------------------------


Check for overfitting

In [153]:
# Investigating Overfitting: Comparing Train vs Validation Accuracy
models = [("Logistic Regression", log_model), ("Random Forest", best_rf)]

print("--- OVERFITTING INVESTIGATION ---")
for name, model in models:
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))
    diff = train_acc - val_acc

    print(f"{name}:")
    print(f"  Training Accuracy:   {train_acc:.4f}")
    print(f"  Validation Accuracy: {val_acc:.4f}")
    print(f"  Gap:                 {diff:.4f}")
    if diff > 0.10:
        print("  Status: Potential Overfitting (Gap > 10%)")
    else:
        print("  Status: Model is Robust (Low Gap)")
    print("-" * 30)

--- OVERFITTING INVESTIGATION ---
Logistic Regression:
  Training Accuracy:   0.7872
  Validation Accuracy: 0.7822
  Gap:                 0.0050
  Status: Model is Robust (Low Gap)
------------------------------
Random Forest:
  Training Accuracy:   0.8342
  Validation Accuracy: 0.7872
  Gap:                 0.0469
  Status: Model is Robust (Low Gap)
------------------------------
